# Muhtemel Ask - 02 ALIGN + PREPARE ID (V2 BETA)
Validate the text-only Turkish correction, resolve only the bounded hash-bound WAV candidates in Colab, force-align the corrected Turkish against audio, re-check independent speech coverage, lock the V2 schema, and publish the Indonesian-only translation pack.

In [ ]:
EPISODE = 12  # @param {type:"integer"}
AUDIO_REVIEW_DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
AUDIO_REVIEW_MODEL = "large-v3"  # @param {type:"string"}
FORCE_AUDIO_REVIEW = False  # @param {type:"boolean"}
MANUAL_REVIEW_PAGE = 1  # @param {type:"integer"}
MANUAL_AUDIO_REVIEW = {}  # Fill only UIDs shown by the bounded review cell.
ALIGN_DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
ALIGNMENT_MODEL = "mpoyraz/wav2vec2-xls-r-300m-cv7-turkish"  # @param {type:"string"}
MIN_WORD_SCORE = 0.30  # @param {type:"number"}
MAX_WORD_DURATION_MS = 2500  # @param {type:"integer"}
MAX_OUTWARD_DRIFT_MS = 500  # @param {type:"integer"}
ALIGNMENT_PADDING_MS = 900  # @param {type:"integer"}
ID_BATCH_SIZE = 400  # @param {type:"integer"}
FORCE_REALIGN = False  # @param {type:"boolean"}

if isinstance(EPISODE, bool) or not isinstance(EPISODE, int) or EPISODE < 1:
    raise ValueError("EPISODE must be a positive integer")
if AUDIO_REVIEW_DEVICE not in {"auto", "cuda", "cpu"}:
    raise ValueError("AUDIO_REVIEW_DEVICE must be auto, cuda, or cpu")
if not AUDIO_REVIEW_MODEL.strip():
    raise ValueError("AUDIO_REVIEW_MODEL cannot be empty")
if isinstance(MANUAL_REVIEW_PAGE, bool) or not isinstance(MANUAL_REVIEW_PAGE, int) or MANUAL_REVIEW_PAGE < 1:
    raise ValueError("MANUAL_REVIEW_PAGE must be a positive integer")
if not isinstance(MANUAL_AUDIO_REVIEW, dict):
    raise ValueError("MANUAL_AUDIO_REVIEW must be a dictionary")
if ALIGN_DEVICE not in {"auto", "cuda", "cpu"}:
    raise ValueError("ALIGN_DEVICE must be auto, cuda, or cpu")
if not ALIGNMENT_MODEL.strip():
    raise ValueError("ALIGNMENT_MODEL cannot be empty")
if isinstance(MIN_WORD_SCORE, bool) or not isinstance(MIN_WORD_SCORE, (int, float)) or not 0.30 <= MIN_WORD_SCORE <= 1:
    raise ValueError("MIN_WORD_SCORE must be a number within [0.30, 1]")
MIN_WORD_SCORE = float(MIN_WORD_SCORE)
if isinstance(MAX_WORD_DURATION_MS, bool) or not isinstance(MAX_WORD_DURATION_MS, int) or not 1 <= MAX_WORD_DURATION_MS <= 2500:
    raise ValueError("MAX_WORD_DURATION_MS must be between 1 and 2500")
if isinstance(MAX_OUTWARD_DRIFT_MS, bool) or not isinstance(MAX_OUTWARD_DRIFT_MS, int) or not 0 <= MAX_OUTWARD_DRIFT_MS <= 500:
    raise ValueError("MAX_OUTWARD_DRIFT_MS must be between 0 and 500")
if isinstance(ALIGNMENT_PADDING_MS, bool) or ALIGNMENT_PADDING_MS < 0:
    raise ValueError("ALIGNMENT_PADDING_MS must be non-negative")
if isinstance(ID_BATCH_SIZE, bool) or not 1 <= ID_BATCH_SIZE <= 1000:
    raise ValueError("ID_BATCH_SIZE must be between 1 and 1000")

## Mount Drive and install the pinned V2 runtime

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import importlib
import shutil
import subprocess
import sys

SYSTEM_ROOT = Path("/content/drive/MyDrive/Muhtemel_Ask_Subtitles/SYSTEM_V2_BETA")
REQUIREMENTS_PATH = SYSTEM_ROOT / "requirements-v2-colab.txt"
if not (SYSTEM_ROOT / "src").is_dir() or not REQUIREMENTS_PATH.is_file():
    raise FileNotFoundError("The complete V2 SYSTEM folder is missing from Drive")
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS_PATH)],
    check=True,
)
if str(SYSTEM_ROOT) not in sys.path:
    sys.path.insert(0, str(SYSTEM_ROOT))
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
importlib.invalidate_caches()
print("Pinned V2 runtime ready.")

## Resolve and validate the exact episode inputs

In [ ]:
import json
import yaml

from src.download import sha256_file
from src.raw_asr_v2 import load_valid_raw_asr_v2
from src.tr_correction import (
    compute_input_sha256,
    read_tr_correction_pack,
    validate_tr_correction_output,
)

with (SYSTEM_ROOT / "config/series.yaml").open(encoding="utf-8") as handle:
    series_config = yaml.safe_load(handle)
EPISODE_NAME = f"Muhtemel Ask {EPISODE}.Bolum"
EPISODE_ROOT = Path(series_config["drive_root"]) / "EPISODES" / EPISODE_NAME
DIRS = {name: EPISODE_ROOT / name for name in (
    "source", "prepare", "translation_input", "translation_output"
)}
if not EPISODE_ROOT.is_dir() or EPISODE_ROOT.is_symlink():
    raise FileNotFoundError(f"Exact episode folder is missing or unsafe: {EPISODE_ROOT}")
for name, folder in DIRS.items():
    if not folder.is_dir() or folder.is_symlink():
        raise FileNotFoundError(f"Required episode directory is missing or unsafe: {name}")

RAW_ASR_PATH = DIRS["prepare"] / "raw_asr_v2.json"
AUDIO_PATH = DIRS["prepare"] / "audio.flac"
TR_CORRECTION_PACK_PATH = (
    DIRS["translation_input"] / f"{EPISODE_NAME}_TR_CORRECTION_PACK.zip"
)
TR_TEXT_CORRECTION_OUTPUT_PATH = (
    DIRS["translation_output"] / f"{EPISODE_NAME}_TR_TEXT_CORRECTED.zip"
)
TR_CORRECTION_OUTPUT_PATH = (
    DIRS["translation_output"] / f"{EPISODE_NAME}_TR_CORRECTED.zip"
)
AUDIO_REVIEW_REPORT_PATH = DIRS["prepare"] / "audio_review_v2.json"
AUDIO_REVIEW_RECOVERY_PATH = DIRS["prepare"] / "audio_review_v2.recovery.json"
FORCED_ALIGNMENT_PATH = DIRS["prepare"] / "forced_alignment_v2.json"
ALIGNMENT_MARKER_PATH = DIRS["prepare"] / "forced_alignment_v2.done.json"
WINDOW_AUDIT_PATH = DIRS["prepare"] / "alignment_window_audit_v2.json"
COVERAGE_PATH = DIRS["prepare"] / "final_speech_coverage_v2.json"
SCHEMA_PATH = DIRS["prepare"] / "aligned_tr_schema_v2.json"
TIMING_QA_PATH = DIRS["prepare"] / "pre_id_timing_qa_v2.json"
ID_PACK_PATH = (
    DIRS["translation_input"] / f"{EPISODE_NAME}_ID_TRANSLATION_PACK.zip"
)
PIPELINE_MARKER_PATH = DIRS["prepare"] / "align_prepare_id_v2.done.json"

def require_exact_file(path, label):
    if path.is_symlink() or not path.is_file() or path.stat().st_size <= 0:
        raise FileNotFoundError(f"{label} is missing, empty, or unsafe: {path}")
    return path

for path, label in (
    (RAW_ASR_PATH, "V2 raw ASR"),
    (AUDIO_PATH, "episode audio"),
    (TR_CORRECTION_PACK_PATH, "exact Turkish correction pack"),
    (TR_TEXT_CORRECTION_OUTPUT_PATH, "exact text-only Turkish correction output"),
):
    require_exact_file(path, label)

raw_asr_data = load_valid_raw_asr_v2(
    DIRS["prepare"],
    audio_path=AUDIO_PATH,
    episode=EPISODE,
    require_independent_vad=True,
)
if raw_asr_data is None:
    raise RuntimeError("V2 raw ASR artifact/marker/audio validation failed")
if raw_asr_data.get("format_version") != "2.0" or raw_asr_data.get("episode") != EPISODE:
    raise RuntimeError("raw_asr_v2.json belongs to another pipeline or episode")
if raw_asr_data.get("independent_vad") is not True:
    raise RuntimeError("raw_asr_v2.json has no independent VAD evidence")
if Path(str(raw_asr_data.get("audio_path", ""))).resolve() != AUDIO_PATH.resolve():
    raise RuntimeError("raw_asr_v2.json is not bound to the exact episode audio path")
audio_sha256 = sha256_file(AUDIO_PATH)
if raw_asr_data.get("audio_sha256") != audio_sha256:
    raise RuntimeError("Episode audio hash differs from raw_asr_v2.json")
expected_correction_input_sha = compute_input_sha256(
    raw_asr_data.get("correction_utterances", []),
    raw_asr_data.get("speech_hole_records", []),
    episode=EPISODE,
    asr_hallucination_records=raw_asr_data.get("asr_hallucination_records", []),
)
correction_pack = read_tr_correction_pack(
    TR_CORRECTION_PACK_PATH,
    expected_input_sha256=expected_correction_input_sha,
)
if correction_pack.manifest.get("episode") != EPISODE:
    raise RuntimeError("Turkish correction pack episode mismatch")
text_correction_output = validate_tr_correction_output(
    TR_CORRECTION_PACK_PATH, TR_TEXT_CORRECTION_OUTPUT_PATH
)
print(f"Validated text-corrected Turkish records: {len(text_correction_output.records):,}")

## Resolve only the bounded review WAVs in Colab
The text model never decides audio. This resumable cell decodes only the hash-bound short clips. Ambiguous items are displayed for explicit listening and cannot pass automatically.

In [ ]:
import zipfile
from IPython.display import Audio, Markdown, display

from src.audio_review_v2 import (
    AudioReviewV2Config,
    AudioReviewV2Error,
    resolve_tr_audio_reviews_v2,
    validate_audio_review_v2_report,
)

audio_review_config = AudioReviewV2Config(
    model_name=AUDIO_REVIEW_MODEL,
    device=AUDIO_REVIEW_DEVICE,
)
try:
    audio_review_data = resolve_tr_audio_reviews_v2(
        TR_CORRECTION_PACK_PATH,
        TR_TEXT_CORRECTION_OUTPUT_PATH,
        TR_CORRECTION_OUTPUT_PATH,
        AUDIO_REVIEW_REPORT_PATH,
        AUDIO_REVIEW_RECOVERY_PATH,
        config=audio_review_config,
        manual_overrides=MANUAL_AUDIO_REVIEW,
        force=FORCE_AUDIO_REVIEW,
    )
except AudioReviewV2Error as exc:
    if AUDIO_REVIEW_REPORT_PATH.is_file():
        with AUDIO_REVIEW_REPORT_PATH.open(encoding="utf-8") as handle:
            pending_report = json.load(handle)
        pending = [item for item in pending_report.get("outcomes", []) if pending_report.get("status") == "NEEDS_MANUAL_REVIEW" and pending_report.get("correction_input_sha256") == correction_pack.manifest["input_sha256"] and pending_report.get("provisional_output_sha256") == text_correction_output.output_sha256 and item.get("decision") == "pending_audio_review"]
        page_size = 10
        page_start = (MANUAL_REVIEW_PAGE - 1) * page_size
        page = pending[page_start:page_start + page_size]
        if page:
            display(Markdown(f"### Manual audio remainder {page_start + 1}-{page_start + len(page)} / {len(pending)}"))
            display(Markdown("Listen to the target interval shown. For speech use `confirmed_dialogue` with the exact Turkish in `tr_corrected`. For no separate dialogue use `reviewed_non_dialogue` on a speech hole or `discarded_asr_hallucination` on an ASR/caption candidate, with empty `tr_corrected`. Add a concrete listening `note`."))
            with zipfile.ZipFile(TR_CORRECTION_PACK_PATH, "r") as archive:
                for item in page:
                    uid = item["utterance_uid"]
                    target_start = item["start_ms"] - item["clip_start_ms"]
                    target_end = item["end_ms"] - item["clip_start_ms"]
                    display(Markdown(f"**{uid}** - {item['evidence_kind']} - target {target_start}-{target_end} ms - secondary: `{item.get('secondary_transcript', '')}`"))
                    display(Audio(data=archive.read(item["audio_member"]), autoplay=False))
            print("Fill MANUAL_AUDIO_REVIEW for these exact UIDs, then rerun this cell.")
    raise RuntimeError(str(exc)) from exc

correction_output = validate_tr_correction_output(
    TR_CORRECTION_PACK_PATH, TR_CORRECTION_OUTPUT_PATH
)
validated_audio_review = validate_audio_review_v2_report(
    TR_CORRECTION_PACK_PATH,
    TR_TEXT_CORRECTION_OUTPUT_PATH,
    TR_CORRECTION_OUTPUT_PATH,
    AUDIO_REVIEW_REPORT_PATH,
)
if validated_audio_review != audio_review_data:
    raise RuntimeError("Audio-review report changed during read-back")
input_file_snapshot = {
    str(path): sha256_file(path)
    for path in (
        RAW_ASR_PATH, AUDIO_PATH, TR_CORRECTION_PACK_PATH,
        TR_TEXT_CORRECTION_OUTPUT_PATH, TR_CORRECTION_OUTPUT_PATH,
        AUDIO_REVIEW_REPORT_PATH,
    )
}
print(f"Bounded audio decisions: {audio_review_data['review_count']:,}")
print(f"Audio-review SHA-256: {audio_review_data['audio_review_sha256']}")
print(f"Final corrected Turkish records: {len(correction_output.records):,}")

## Derive padded acoustic-search windows with the V2 pipeline

In [ ]:
import torch

from src.download import sha256_json
from src.v2_pipeline import (
    build_strict_v2_artifacts,
    correction_records_to_alignment_inputs,
    create_v2_id_translation_pack,
)

alignment_bundle = correction_records_to_alignment_inputs(
    correction_pack.utterances,
    correction_output.records,
    speech_hole_records=correction_pack.speech_holes,
    asr_hallucination_records=correction_pack.asr_hallucination_records,
    alignment_padding_ms=ALIGNMENT_PADDING_MS,
)
if not alignment_bundle.alignment_inputs:
    raise RuntimeError("No corrected dialogue remains for forced alignment")
alignment_device = (
    "cuda" if ALIGN_DEVICE == "auto" and torch.cuda.is_available() else
    "cpu" if ALIGN_DEVICE == "auto" else ALIGN_DEVICE
)
if alignment_device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but this runtime has no compatible GPU")

alignment_code_paths = (
    SYSTEM_ROOT / "src/audio_review_v2.py",
    SYSTEM_ROOT / "src/forced_align.py",
    SYSTEM_ROOT / "requirements-v2-colab.txt",
)
alignment_code_sha256 = {
    str(path.relative_to(SYSTEM_ROOT)): sha256_file(path)
    for path in alignment_code_paths
}
alignment_identity = {
    "audio_sha256": audio_sha256,
    "correction_output_sha256": correction_output.output_sha256,
    "audio_review_sha256": audio_review_data["audio_review_sha256"],
    "alignment_inputs": alignment_bundle.alignment_inputs,
    "reviewed_dialogue": alignment_bundle.reviewed_dialogue,
    "discarded_asr_hallucinations": alignment_bundle.discarded_asr_hallucinations,
    "alignment_padding_ms": ALIGNMENT_PADDING_MS,
    "model_name": ALIGNMENT_MODEL,
    "min_word_score": MIN_WORD_SCORE,
    "max_word_duration_ms": MAX_WORD_DURATION_MS,
    "max_outward_drift_ms": MAX_OUTWARD_DRIFT_MS,
    "device": alignment_device,
    "code_sha256": alignment_code_sha256,
}
alignment_input_sha256 = sha256_json(alignment_identity)
print(f"Alignment windows: {len(alignment_bundle.alignment_inputs):,}")
print(f"Reviewed non-dialogue intervals: {len(alignment_bundle.reviewed_non_dialogue):,}")
print(f"Reviewed dialogue exceptions: {len(alignment_bundle.reviewed_dialogue):,}")
print(f"Discarded ASR/caption hallucinations: {len(alignment_bundle.discarded_asr_hallucinations):,}")
print(f"Alignment device: {alignment_device}")

## Force-align corrected Turkish without interpolation

In [ ]:
import gc

from src.download import (
    atomic_write_json,
    load_valid_stage_marker,
    write_stage_marker,
)
from src.forced_align import align_corrected_segments, validate_forced_alignment_data

def validate_alignment_checkpoint(data):
    validate_forced_alignment_data(data)
    if data.get("audio_sha256") != audio_sha256:
        raise RuntimeError("Forced alignment belongs to stale or different audio")
    persisted_inputs = tuple({
        "start_ms": segment["alignment_window_start_ms"],
        "end_ms": segment["alignment_window_end_ms"],
        "text": segment["text"],
        "asr_text": segment["asr_text"],
        "deletion_audio_reviewed": segment["deletion_audio_reviewed"],
        "utterance_uid": segment["utterance_uid"],
        "coarse_start_ms": segment["coarse_start_ms"],
        "coarse_end_ms": segment["coarse_end_ms"],
    } for segment in data["segments"])
    if persisted_inputs != alignment_bundle.alignment_inputs:
        raise RuntimeError("Forced alignment belongs to different corrected text/windows")
    return data

alignment_marker = None if FORCE_REALIGN else load_valid_stage_marker(
    ALIGNMENT_MARKER_PATH,
    stage="forced_alignment_v2",
    input_sha256=alignment_input_sha256,
    required_output_keys=("forced_alignment_v2",),
    allowed_root=DIRS["prepare"],
)
forced_alignment = None
if alignment_marker is not None:
    try:
        with FORCED_ALIGNMENT_PATH.open(encoding="utf-8") as handle:
            forced_alignment = json.load(handle)
        validate_alignment_checkpoint(forced_alignment)
    except Exception:
        forced_alignment = None
if forced_alignment is None:
    forced_alignment = align_corrected_segments(
        AUDIO_PATH,
        alignment_bundle.alignment_inputs,
        language="tr",
        device=alignment_device,
        model_name=ALIGNMENT_MODEL,
        min_word_score=MIN_WORD_SCORE,
        max_word_duration_ms=MAX_WORD_DURATION_MS,
        max_outward_drift_ms=MAX_OUTWARD_DRIFT_MS,
    )
    validate_alignment_checkpoint(forced_alignment)
    atomic_write_json(FORCED_ALIGNMENT_PATH, forced_alignment)
    write_stage_marker(
        ALIGNMENT_MARKER_PATH,
        stage="forced_alignment_v2",
        input_sha256=alignment_input_sha256,
        outputs={"forced_alignment_v2": FORCED_ALIGNMENT_PATH},
        details={
            "audio_sha256": forced_alignment["audio_sha256"],
            "alignment_sha256": forced_alignment["alignment_sha256"],
            "correction_output_sha256": correction_output.output_sha256,
            "model_name": ALIGNMENT_MODEL,
            "min_word_score": MIN_WORD_SCORE,
            "max_word_duration_ms": MAX_WORD_DURATION_MS,
            "max_outward_drift_ms": MAX_OUTWARD_DRIFT_MS,
            "device": alignment_device,
        },
    )
    print("Forced alignment completed.")
else:
    print("Hash-validated forced alignment resumed.")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"Alignment SHA-256: {forced_alignment['alignment_sha256']}")

## Re-check independent VAD coverage and lock the strict pre-ID timeline

In [ ]:
from src.id_translation import load_default_id_translation_glossary
from src.timing_qa_v2 import assert_timing_qa_v2

artifacts = build_strict_v2_artifacts(
    raw_asr_data,
    correction_output.records,
    forced_alignment,
    episode=EPISODE,
    alignment_padding_ms=ALIGNMENT_PADDING_MS,
    acoustic_audio_review=audio_review_data,
)
if artifacts.preparation != alignment_bundle:
    raise RuntimeError("V2 alignment preparation changed between validation passes")
def require_authoritative_coverage(artifacts_value):
    report = artifacts_value.speech_coverage_report
    audio_review = report.get("audio_review_v2")
    strict_word_vad = report.get("strict_word_vad_v2")
    if (
        report.get("status") != "PASS"
        or report.get("unresolved_speech_region_count") != 0
        or not isinstance(audio_review, dict)
        or audio_review.get("status") != "PASS"
        or audio_review.get("pending_audio_review_count") != 0
        or not isinstance(strict_word_vad, dict)
        or strict_word_vad.get("status") != "PASS"
        or strict_word_vad.get("unsafe_word_count") != 0
        or strict_word_vad.get("confirmed_dialogue_coverage_fail_count") != 0
    ):
        raise RuntimeError("Authoritative independent-VAD/audio-review gate failed")
    return report

final_speech_coverage = require_authoritative_coverage(artifacts)
if artifacts.schema.get("alignment_sha256") != forced_alignment["alignment_sha256"]:
    raise RuntimeError("Aligned schema is not bound to the forced alignment digest")
assert_timing_qa_v2(artifacts.timing_qa_report)
id_glossary = load_default_id_translation_glossary()
pipeline_code_paths = (
    SYSTEM_ROOT / "src/audio_review_v2.py",
    SYSTEM_ROOT / "src/v2_pipeline.py",
    SYSTEM_ROOT / "src/segment_v2.py",
    SYSTEM_ROOT / "src/schema_v2.py",
    SYSTEM_ROOT / "src/speech_coverage.py",
    SYSTEM_ROOT / "src/timing_qa_v2.py",
    SYSTEM_ROOT / "src/id_translation.py",
    SYSTEM_ROOT / "config/names.yaml",
    SYSTEM_ROOT / "config/religious_terms.yaml",
)
pipeline_code_sha256 = {
    str(path.relative_to(SYSTEM_ROOT)): sha256_file(path)
    for path in pipeline_code_paths
}
pipeline_input_sha256 = sha256_json({
    "alignment_sha256": forced_alignment["alignment_sha256"],
    "correction_output_sha256": correction_output.output_sha256,
    "audio_review_sha256": audio_review_data["audio_review_sha256"],
    "audio_sha256": audio_sha256,
    "alignment_padding_ms": ALIGNMENT_PADDING_MS,
    "min_word_score": MIN_WORD_SCORE,
    "max_word_duration_ms": MAX_WORD_DURATION_MS,
    "max_outward_drift_ms": MAX_OUTWARD_DRIFT_MS,
    "id_batch_size": ID_BATCH_SIZE,
    "id_glossary_sha256": sha256_json(id_glossary),
    "code_sha256": pipeline_code_sha256,
})
print(f"V2 subtitle blocks: {artifacts.schema['block_count']:,}")
print(f"V2 schema SHA-256: {artifacts.schema['schema_sha256']}")
print("Pre-ID timing QA: PASS")

## Publish the exact aligned schema and Indonesian-only pack

In [ ]:
from src.id_translation import validate_id_translation_pack

atomic_write_json(WINDOW_AUDIT_PATH, list(alignment_bundle.window_audit))
atomic_write_json(COVERAGE_PATH, final_speech_coverage)
atomic_write_json(SCHEMA_PATH, artifacts.schema)
atomic_write_json(TIMING_QA_PATH, artifacts.timing_qa_report)
current_input_snapshot = {path: sha256_file(Path(path)) for path in input_file_snapshot}
if current_input_snapshot != input_file_snapshot:
    raise RuntimeError("A raw/audio/TR input changed during alignment; refusing publication")
id_pack_manifest = create_v2_id_translation_pack(
    artifacts, ID_PACK_PATH, batch_size=ID_BATCH_SIZE, glossary=id_glossary
)
validated_id_manifest = validate_id_translation_pack(
    ID_PACK_PATH, expected_schema=artifacts.schema, expected_glossary=id_glossary
)
if validated_id_manifest != id_pack_manifest:
    raise RuntimeError("Published ID translation pack failed deterministic read-back")
write_stage_marker(
    PIPELINE_MARKER_PATH,
    stage="align_prepare_id_v2",
    input_sha256=pipeline_input_sha256,
    outputs={
        "forced_alignment_v2": FORCED_ALIGNMENT_PATH,
        "audio_review_v2": AUDIO_REVIEW_REPORT_PATH,
        "alignment_window_audit_v2": WINDOW_AUDIT_PATH,
        "final_speech_coverage_v2": COVERAGE_PATH,
        "aligned_tr_schema_v2": SCHEMA_PATH,
        "pre_id_timing_qa_v2": TIMING_QA_PATH,
        "id_translation_pack_v2": ID_PACK_PATH,
    },
    details={
        "episode": EPISODE,
        "audio_sha256": forced_alignment["audio_sha256"],
        "alignment_sha256": forced_alignment["alignment_sha256"],
        "audio_review_sha256": audio_review_data["audio_review_sha256"],
        "schema_sha256": artifacts.schema["schema_sha256"],
        "block_count": artifacts.schema["block_count"],
    },
)
print(f"ID translation pack ready: {ID_PACK_PATH}")
print("Send only this exact ID pack for Indonesian translation.")